In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from food_cv.classifier import build_resnet50_classifier
from food_cv.config import ProjectPaths
from food_cv.data_pipeline import DataConfig, Food101DataModule
from food_cv.pipeline import MealPredictor
from food_cv.training import TrainConfig, train_classifier

paths = ProjectPaths.from_root(PROJECT_ROOT)
candidate_data_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT,
]
data_root = next((p for p in candidate_data_roots if p.exists()), PROJECT_ROOT)
data_config = DataConfig(data_root=data_root, batch_size=32, num_workers=2, image_size=224)

try:
    datamodule = Food101DataModule(data_config)
    train_loader, val_loader, test_loader = datamodule.build_dataloaders()
    print(f"train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")
except Exception as e:
    train_loader = val_loader = test_loader = None
    print(f"Data loader 初始化失败: {e}")

model = build_resnet50_classifier(num_classes=101, freeze_backbone=True)
checkpoint_path = paths.model_dir / "food101_resnet50_fc_only.pt"

if train_loader is not None and val_loader is not None:
    metrics = train_classifier(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        save_path=checkpoint_path,
        config=TrainConfig(epochs=1, lr=1e-4, device="cuda"),
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("跳过训练：请确认 Food-101 已下载并放到 data 目录")

predictor = MealPredictor(paths=paths, checkpoint_path=checkpoint_path if checkpoint_path.exists() else None)
demo_image = PROJECT_ROOT / "demo.jpg"
if demo_image.exists():
    result = predictor.predict_meal(demo_image)
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("未找到 demo.jpg，推理演示已跳过")
